# R47 - expanded supplementary-data pilot

**Choose a GPU runtime, then Run all.** The notebook uses four code cells.
The first two perform CPU preparation. A T4 completed the prior R45 recipe;
this notebook checks the actual GPU/runtime and runs a discarded resource
trial before the two full arms.

Reference: the original 137 photos with effective R37 labels, plus the
exact frame_0383 labels used by PR133. Candidate: the same original 137
photos, plus all complete R47 photos: `frame_0146`, `frame_0160`, `frame_0178`, `frame_0371`, `frame_0383`.
Shared frame_0383 labels are byte-identical in both arms.

| Planned training exposure | Reference | Candidate |
| --- | ---: | ---: |
| Photographs | 138 | 142 |
| Fronts | 3827 | 3999 |
| Locatable corner slots | 6923 | 7259 |
| Unavailable corner slots | 731 | 739 |
| Epochs | 30 | 30 |
| Batches | 1050 | 1080 |
| Expected measured optimizer updates | 78 | 80 |
| Image presentations | 4140 | 4260 |

This is an **expanded-data, equal-epoch pilot**. Additional exposure is part
of the treatment; the experiment does not isolate new data from compute.
Both arms start independently from epoch173 with the same seed46 recipe.
Held photographs are excluded as whole images. The original 20 development
photos and all three evaluation references remain unchanged.

Keep the downloaded return ZIP. Optional Drive backups preserve verified
completed arms. Interrupted runs remain intact; no partial optimizer resume
or automatic model promotion occurs. PR35 remains selected pending review.


## 1. Load the published source

The small notebook points to one immutable source commit. It preserves an
incompatible existing checkout; choose a fresh base folder when necessary.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

SOURCE_REVISION = "96b05f0d2149ab71c9211c81f35be67f79344b81"
RELEASE_SHA256 = "1d4f3ac4cb3efc97c6e05ef392b83654d4eed2039f3f167340a425ff837641a7"
BASE = Path(os.environ.get("BAOMAI_R47_BASE", "/content/baomai-r47")).resolve()
SOURCE, WORK, CACHE = BASE / "source", BASE / "work", BASE / "cache"
REVIEW = WORK / "review"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def sha(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Preserved nonempty source folder; choose a fresh BAOMAI_R47_BASE")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_REVISION], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "/scripts/", "/data/manifest.json", "/data/labels/",
         "/data/candidates/front-review-2026-11-26-r37/", "/data/reviews/fronts-2026-11-26-r38/",
         "/data/reviews/additional-fronts-2026-11-27-r46/", "/data/reviews/additional-fronts-2026-11-27-r47/",
         "/data/references/sequence-refined-reference-2026-11-18/",
         "/data/references/visible-front-completion-2026-11-14/"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_REVISION], SOURCE)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=SOURCE, text=True)
if head != SOURCE_REVISION or dirty:
    raise RuntimeError("Source differs; preserve it and choose a fresh BAOMAI_R47_BASE")
assert sha(SOURCE / "data/reviews/additional-fronts-2026-11-27-r47/release-status.json") == RELEASE_SHA256
SCRIPT = SOURCE / "scripts/run_additional_front_pilot_r47.py"
def command(action, *extra):
    return [sys.executable, "-B", SCRIPT, action, "--root", SOURCE,
            "--work", WORK, "--review", REVIEW, *extra]
print("Pinned source:", head)


## 2. Verify labels and restore inputs

This CPU step replays the R47 review, checks the complete-photo export and
freezes both treatments. The cache downloads only the 157 original PNGs,
epoch173 and the pinned review archive. Every payload is hash checked.
It does not install an ML framework or train a model.


In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "Pillow==13.0.0", "reportlab==4.4.9"])
run([sys.executable, "-B", SOURCE / "scripts/build_front_review_r47.py",
     "--r44-archive", CACHE / "r44-review.zip", "--download-r44", "--output", REVIEW])
run(command("prepare", "--inputs-dir", CACHE / "inputs", "--download-inputs"))
PLAN = json.loads((WORK / "staging.json").read_text())
print("Complete candidate photos:", PLAN["treatment"]["candidate_extra_ids"])
for role, budget in PLAN["budgets"].items():
    print(role, PLAN["counts"][role]["train"], "batches", budget["batches"],
          "updates", budget["steps"], "image presentations", budget["images"])
print("CPU preflight complete. No training has occurred.")


## 3. Resource trial and both full arms

The actual native training source, dataset loader, coordinates, image
exposure and optimizer-update schedule are checked. Both full arms use
fresh epoch173; the four-batch trial is discarded. Verified completed arms
can be recovered from Drive. Set `USE_DRIVE = False` to skip that backup.


In [ ]:
from importlib.metadata import version
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before this cell."
print("GPU:", torch.cuda.get_device_name(0))
constraints = BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")))
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
USE_DRIVE = True
BACKUP = None
if USE_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        print("Drive unavailable; keep the final downloaded ZIP.")
    else:
        drive.mount("/content/drive")
        BACKUP = Path("/content/drive/MyDrive/baomai-r47") / SOURCE_REVISION
        BACKUP.mkdir(parents=True, exist_ok=True)
extra = ["--backup-dir", BACKUP] if BACKUP else []
run(command("run-pair", *extra))


## 4. Score and download the complete return

Inference uses unchanged operating points. The return contains both final
checkpoints, receipts, prediction caches, exact labels, source closure and
review provenance. Existing cached evaluation and return files are verified
before reuse. Counts are reported separately for current R38, frozen666 and
historical663; they describe the same development photographs.


In [ ]:
ARCHIVE = BASE / "baomai-additional-front-pilot-r47-results.zip"
available_return = ARCHIVE if ARCHIVE.exists() else (
    BACKUP / ARCHIVE.name if BACKUP and (BACKUP / ARCHIVE.name).exists() else None)
if available_return:
    run(command("restore-return", "--output", available_return))
    if available_return != ARCHIVE:
        shutil.copyfile(available_return, ARCHIVE)
        assert sha(available_return) == sha(ARCHIVE)
else:
    complete_evaluation = all((WORK / "evaluation" / name).is_file()
                              for name in ("summary.json", "predictions.json"))
    run(command("verify-evaluation" if complete_evaluation else "evaluate"))
run(command("package", "--output", ARCHIVE))
summary = json.loads((WORK / "evaluation/summary.json").read_text())
for reference in ("r38", "frozen666", "historical663"):
    for role in ("reference", "candidate"):
        report = summary["references"][reference]["postclip"][role]
        d, c = report["detection"], report["corners_by_locatable_count"]["2"]
        print(reference, role, "misses", d["fn"], "unmatched", d["fp"],
              "duplicate subset", d["duplicate_fp"], "other", d["other_fp"],
              "both within5px", c["within5"], "/", c["eligible"],
              "within10px", c["within10"])
if BACKUP:
    destination = BACKUP / ARCHIVE.name
    if destination.exists():
        assert sha(destination) == sha(ARCHIVE), "Different completed return preserved"
    else:
        shutil.copyfile(ARCHIVE, destination)
    assert sha(destination) == sha(ARCHIVE)
print("Return:", ARCHIVE, "SHA256:", sha(ARCHIVE))
try:
    from google.colab import files
except ImportError:
    print("Saved locally:", ARCHIVE)
else:
    files.download(str(ARCHIVE))
